In [2]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage

import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

import aiosqlite
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

load_dotenv()

model = init_chat_model(
    model="deepseek-chat"
)

conn = sqlite3.connect("data.sqlite", check_same_thread=False)
checkpointer = SqliteSaver(conn)

agent = create_agent(
    model=model,
    system_prompt=SystemMessage("你是一个全能助手！"),
    checkpointer=checkpointer
)

thread_id = str(1)
config = {"configurable": {"thread_id": thread_id}}

response1 = agent.invoke(
    {"messages": [HumanMessage("你好，我是小明")]},
    config=config
)

response2 = agent.invoke(
    {"messages": [HumanMessage("我叫什么名字")]},
    config=config
)

In [4]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
import uuid

load_dotenv()

model = init_chat_model(
    model="deepseek-chat"
)

@tool
def get_beijing_weather() -> str:
    """Get the current weather in beijing."""
    return "20 degrees Celsius"

@tool
def get_shanghai_weather() -> str:
    """Get the current weather in shanghai."""
    return "25 degrees Celsius"

agent = create_agent(
    model=model,
    tools=[get_beijing_weather, get_shanghai_weather],
    system_prompt=SystemMessage("You are a helpful assistant."),
    checkpointer=InMemorySaver()
)

thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

response1 = agent.invoke(
    {"messages": [HumanMessage("What is the weather in beijing and shanghai?")]},
    config=config
)

for message in response1["messages"]:
    message.pretty_print()



================================ Human Message =================================

What is the weather in beijing and shanghai?
================================== Ai Message ==================================

I'll get the current weather for both Beijing and Shanghai for you.
Tool Calls:
  get_beijing_weather (call_00_6VAXvXkq2SaMWtZPOCCStGUL)
 Call ID: call_00_6VAXvXkq2SaMWtZPOCCStGUL
  Args:
  get_shanghai_weather (call_01_GnJMJkfhGoeI8TlJv6VKGWoK)
 Call ID: call_01_GnJMJkfhGoeI8TlJv6VKGWoK
  Args:
================================= Tool Message =================================
Name: get_beijing_weather

20 degrees Celsius
================================= Tool Message =================================
Name: get_shanghai_weather

25 degrees Celsius
================================== Ai Message ==================================

Here's the current weather in both cities:

- **Beijing**: 20°C
- **Shanghai**: 25°C

Shanghai is currently warmer than Beijing, with a 5-degree difference.


In [5]:
from langchain.messages import AIMessage


for message in response1["messages"]:
    if isinstance(message, AIMessage):
        print(message.tool_calls)

[{'name': 'get_beijing_weather', 'args': {}, 'id': 'call_00_6VAXvXkq2SaMWtZPOCCStGUL', 'type': 'tool_call'}, {'name': 'get_shanghai_weather', 'args': {}, 'id': 'call_01_GnJMJkfhGoeI8TlJv6VKGWoK', 'type': 'tool_call'}]
[]


In [4]:
for message in response2["messages"]:
    message.pretty_print()

================================ Human Message =================================

你好，我是小明
================================== Ai Message ==================================

你好，小明！很高兴认识你！😊  
有什么我可以帮助你的吗？无论是学习、工作、生活上的问题，还是想聊聊天，我都在这里哦！
================================ Human Message =================================

我叫什么名字
================================== Ai Message ==================================

你刚才提到你的名字是**小明**呀！😊  
如果这是个脑筋急转弯或者我记错了，随时告诉我哦～


In [5]:
conn.close()